# AegisFin — Enterprise Banking Customer Intelligence & Transaction Analytics Suite
**Platform:** AegisFin Data Intelligence
**Architecture:** Python 3.11, Pandas, NumPy, SQLite / PostgreSQL, SQL, Machine Learning

---

## 1. Executive Objective & Analytical Scope
This analytical notebook implements an enterprise-grade banking customer intelligence pipeline. It executes end-to-end data cleansing, demographic normalization, Recency-Frequency-Monetary (RFM) quintile segmentation, churn cohort risk modeling, and geographic market penetration analytics across banking transaction portfolios.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from datetime import datetime

# Set display options & aesthetics
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Helvetica'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8

## 2. Ingestion & Raw Data Cleansing Audit
The pipeline ingests raw banking records and normalizes schemas:
1. **Demographic Imputation:** Imputes missing or non-plausible dates of birth using demographic median values.
2. **Location Standardization:** Normalizes 16 urban clusters with uppercase trimming and alias consolidation.
3. **Temporal Serialization:** Converts timestamp records into standardized ISO-8601 UTC formats.

In [2]:
df_cust = pd.read_csv('../data/processed/customers_cleaned.csv')
df_tx = pd.read_csv('../data/processed/transactions_cleaned.csv')
df_rfm = pd.read_csv('../data/processed/customer_rfm.csv')

print(f"Validated Unique Accounts: {len(df_cust):,}")
print(f"Processed Transactions: {len(df_tx):,}")
print(f"Segmented RFM Records: {len(df_rfm):,}")

## 3. Core Banking Key Performance Indicators (KPIs)

In [3]:
total_accounts = df_cust['customer_id'].nunique()
total_tx_count = len(df_tx)
total_gross_volume = df_tx['transaction_amount'].sum()
mean_ticket_size = df_tx['transaction_amount'].mean()
mean_velocity = total_tx_count / total_accounts

print(f"--- AegisFin KPI Executive Telemetry ---")
print(f"Total Unique Accounts: {total_accounts:,}")
print(f"Total Transaction Count: {total_tx_count:,}")
print(f"Gross Processed Volume: ₹{total_gross_volume:,.2f}")
print(f"Mean Transaction Ticket: ₹{mean_ticket_size:,.2f}")
print(f"Mean Customer Velocity: {mean_velocity:.2f} transactions/account")

## 4. RFM Quintile Behavioral Segmentation
Accounts are segmented into distinct commercial behavioral tiers (Champions, Loyal Customers, Big Spenders, Potential Loyalists, At Risk, Dormant / Lost) via quintile discretization.

In [4]:
rfm_summary = df_rfm.groupby('customer_segment').agg(
    Account_Count=('customer_id', 'count'),
    Aggregate_Volume=('monetary', 'sum'),
    Mean_Volume=('monetary', 'mean'),
    Mean_Frequency=('frequency', 'mean'),
    Mean_Dormancy_Days=('recency', 'mean')
).reset_index()

rfm_summary['Account_Share_Pct'] = (rfm_summary['Account_Count'] / len(df_rfm)) * 100
rfm_summary['Volume_Share_Pct'] = (rfm_summary['Aggregate_Volume'] / df_rfm['monetary'].sum()) * 100
rfm_summary = rfm_summary.sort_values(by='Aggregate_Volume', ascending=False)

display(rfm_summary)

## 5. Metropolitan Market Penetration Matrix

In [5]:
metro_distribution = df_tx['location'].value_counts().head(12).reset_index()
metro_distribution.columns = ['Metropolitan_Cluster', 'Transaction_Volume']
display(metro_distribution)

## 6. Strategic Growth & Risk Mitigation Recommendations
1. **High-Yield Pareto Retention:** Prioritize Tier-1 Champions and Loyal Customers with dedicated commercial relationship desks.
2. **Automated Churn Interventions:** Deploy rule-based fee waivers and transaction milestone incentives for accounts entering the 30+ day dormancy cohort.
3. **Metropolitan Credit Expansion:** Capitalize on Tier-1 metro concentration (Mumbai, New Delhi, Bangalore) to deploy tailored credit card and overdraft products.